# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue combines two signals rather than trusting the tree alone,
because Week 6 showed the tree's advantage varies a lot by which clients are
held out (P@50 ranged 0.26-0.80 across 10 draws, averaging 0.60 vs the
baseline's 0.30). A page gets a high combined rank only when both the tree
and the Week 4 rule agree it's worth reviewing; that agreement is itself a
usable reason code.

**Reason codes:**
- `both_flag_high_confidence` — tree and rule both flag this page
- `tree_only` — tree flags it, rule doesn't (use with more caution)
- `rule_only` — rule flags it, tree doesn't
- `neither_flag` — no action


**Finding:** the tree's top 500 and the rule's top 500 share zero pages. The
two methods are not agreeing-and-disagreeing on the same candidates, they are
finding different populations. The rule clusters tightly around position 11
(closest to the "quick-win zone" edge); the tree's picks spread across
positions 11-35 with lower impressions and slightly higher CTR on average.
This means there is currently no `both_flag_high_confidence` tier — a real
limit worth stating plainly, not smoothing over.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
FEATURES = ["avg_position", "total_impressions", "ctr"]

def load_month_features(month):
    path = f"{BASE}/fact_content_daily_performance/month={month}/data_0.parquet"
    df = con.execute(f"""
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_avg_position) AS avg_position,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks) AS total_clicks
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    """).df()
    df["ctr"] = np.where(df["total_impressions"] > 0,
                         df["total_clicks"] / df["total_impressions"], 0.0)
    return df

def build_pair(month_a, month_b, min_impressions=100):
    a = load_month_features(month_a)
    b = load_month_features(month_b)[["content_hash_id", "client_hash_id", "total_clicks"]]
    b = b.rename(columns={"total_clicks": "clicks_next"})
    m = a.merge(b, on=["content_hash_id", "client_hash_id"], how="inner")
    m = m[m["total_impressions"] >= min_impressions].copy()
    m["is_declining"] = (m["clicks_next"] < 0.8 * m["total_clicks"]).astype(int)
    return m

def baseline_score_row(r):
    if 11 <= r["avg_position"] <= 20 and r["total_impressions"] >= 100:
        return 400 - r["avg_position"]
    elif r["avg_position"] <= 10 and r["ctr"] < 0.01 and r["total_impressions"] >= 100:
        return 200 + min(r["total_impressions"] / 1000, 99)
    return min(r["total_impressions"] / 10000, 99)

train_df = build_pair("2026-01", "2026-02")
test_df = build_pair("2026-04", "2026-05")

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(train_df[FEATURES], train_df["is_declining"])

queue = test_df.copy()
queue["tree_score"] = tree.predict_proba(queue[FEATURES])[:, 1]
queue["baseline_score"] = queue.apply(baseline_score_row, axis=1)

tree_top = set(queue.sort_values("tree_score", ascending=False).head(500)["content_hash_id"])
rule_top = set(queue.sort_values("baseline_score", ascending=False).head(500)["content_hash_id"])
overlap = tree_top & rule_top
print(f"Tree top 500 vs rule top 500 overlap: {len(overlap)} pages")

def reason_code(row):
    in_tree = row["content_hash_id"] in tree_top
    in_rule = row["content_hash_id"] in rule_top
    if in_tree and in_rule:
        return "both_flag_high_confidence"
    elif in_tree:
        return "tree_only"
    elif in_rule:
        return "rule_only"
    return "neither_flag"

queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = np.where(queue["reason_code"] == "neither_flag", "no_action", "review_for_refresh")

# order follows the reason code groups directly, not a separate rank sum
priority = {"both_flag_high_confidence": 0, "tree_only": 1, "rule_only": 2, "neither_flag": 3}
queue["priority"] = queue["reason_code"].map(priority)
queue["within_group_score"] = np.where(
    queue["reason_code"].isin(["both_flag_high_confidence", "tree_only"]),
    queue["tree_score"], queue["baseline_score"]
)
ranked_queue = queue.sort_values(["priority", "within_group_score"], ascending=[True, False]).reset_index(drop=True)

print(ranked_queue["reason_code"].value_counts())
ranked_queue[["content_hash_id", "avg_position", "total_impressions", "ctr",
              "reason_code", "action"]].head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tree top 500 vs rule top 500 overlap: 0 pages
reason_code
neither_flag    105586
tree_only          500
rule_only          500
Name: count, dtype: int64


,content_hash_id,avg_position,total_impressions,ctr,reason_code,action
0,content_6c66f0331446c706,14.196213,346.0,0.002890,tree_only,review_for_refresh
1,content_543e7c98eeb71167,23.642885,161.0,0.006211,tree_only,review_for_refresh
2,content_e7092c22c303eade,46.281504,169.0,0.005917,tree_only,review_for_refresh
3,content_6e820cbbe3b4f80e,55.748612,358.0,0.002793,tree_only,review_for_refresh
4,content_fc052d5c1f9c8741,41.112707,387.0,0.002584,tree_only,review_for_refresh
5,content_b7937e15573cfe43,17.347140,247.0,0.004049,tree_only,review_for_refresh
6,content_e9b603d6c98f0b71,23.130754,127.0,0.007874,tree_only,review_for_refresh
7,content_96326d73865c32e5,22.679694,122.0,0.008197,tree_only,review_for_refresh
8,content_04260f7aab4868a9,21.332678,141.0,0.007092,tree_only,review_for_refresh
9,content_ac9e10e79f85ea88,21.812776,109.0,0.036697,tree_only,review_for_refresh


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a weekly triage list for a content editor with limited
review capacity, drawing from `tree_only` and `rule_only` pages. Not a
fully automated refresh trigger.

**Limits:**
- Evaluated on 50 clients total (15 held out per test draw); not yet proven
  at broader scale.
- The tree's precision varied from 0.26 to 0.80 across 10 client draws
  (Week 6 audit) — it is not stable enough to trust on its own.
- Zero overlap between tree and rule picks means there is no high-confidence
  tier yet; every flagged page is single-method evidence, not corroborated.
- The label measures a 20%+ month-over-month click drop, which can come from
  seasonality, not just a fixable content problem.
- 39.6% of pages had zero clicks in the feature month and can never be
  labeled declining by this formula (Week 3/6 finding) — those pages are
  systematically excluded from ever being flagged as `tree_only`.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
limits = {
    "clients_evaluated": 50,
    "tree_precision_range_week6": "0.26 to 0.80 across 10 draws",
    "tree_rule_overlap": 0,
    "zero_click_pages_excluded_pct": round((train_df["total_clicks"] == 0).mean(), 3),
}
for k, v in limits.items():
    print(f"{k}: {v}")

clients_evaluated: 50
tree_precision_range_week6: 0.26 to 0.80 across 10 draws
tree_rule_overlap: 0
zero_click_pages_excluded_pct: 0.396


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a human must check:**
- Whether the click drop lines up with a known seasonal pattern for that
  page/topic (holiday content, event-based queries).
- Whether the page's ranking query set actually matches its content (a
  position/CTR problem is sometimes a targeting mismatch, not a content-decay
  problem).
- Whether the client relationship or contract has specific constraints on
  what content changes are allowed.

**No-go — never automate:**
- Auto-publishing content changes based on the score alone.
- Auto-deprioritizing/removing a page based on the score alone — false
  positives exist on both methods.
- Treating `tree_only` picks as equally trustworthy as a manually reviewed
  finding, given the 0.26-0.80 precision range.
- Using this queue for clients outside the 50 evaluated here without
  re-validating first.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go = ["auto-publish content changes", "auto-remove/deprioritize a page",
         "treat tree_only as equal confidence to manual review",
         "apply to un-evaluated clients without re-validation"]
print("Never automate:")
for item in no_go:
    print(f"- {item}")

Never automate:
- auto-publish content changes
- auto-remove/deprioritize a page
- treat tree_only as equal confidence to manual review
- apply to un-evaluated clients without re-validation


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain/monitor if:**
- Precision@50 on a fresh held-out month drops meaningfully below the 0.60
  average observed in Week 6.
- The zero-overlap pattern between tree and rule changes significantly (would
  suggest the underlying data distribution shifted).
- The zero-click page share moves far from ~40% (would change how much of
  the portfolio the label can even address).
- More than ~90 days pass since the last retrain, given the data itself
  shows drift across months (Week 6 train/test used a 2-month gap already).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
triggers = {
    "precision_floor": 0.60,
    "recheck_cadence_days": 90,
    "zero_click_share_baseline": round((train_df["total_clicks"] == 0).mean(), 3),
}
for k, v in triggers.items():
    print(f"{k}: {v}")

precision_floor: 0.6
recheck_cadence_days: 90
zero_click_share_baseline: 0.396


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked_queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

import json
metrics = {
    "week6_tree_p50_mean": 0.60,
    "week6_tree_p50_min": 0.26,
    "week6_tree_p50_max": 0.80,
    "week6_baseline_p50_mean": 0.30,
    "tree_rule_overlap_top500": 0,
    "zero_click_page_share": round((train_df["total_clicks"] == 0).mean(), 3),
    "clients_evaluated": 50,
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported: work/outputs/action_playbook_queue.csv")
print("Exported: work/outputs/w07_metrics.json")
print(metrics)

Exported: work/outputs/action_playbook_queue.csv
Exported: work/outputs/w07_metrics.json
{'week6_tree_p50_mean': 0.6, 'week6_tree_p50_min': 0.26, 'week6_tree_p50_max': 0.8, 'week6_baseline_p50_mean': 0.3, 'tree_rule_overlap_top500': 0, 'zero_click_page_share': np.float64(0.396), 'clients_evaluated': 50}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.